In [1]:
import pandas as pd
import sqlite3

In [19]:
def download():
    
    from urllib.request import urlretrieve
    import os

    url = (
        "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/"
        "Maji_Ndogo/Maji_Ndogo_farm_survey_small.db"
    )

    db_file = "Maji_Ndogo_farm_survey_small.db"

    # Download only if the database is not already present
    if not os.path.exists(db_file):
        urlretrieve(url, db_file)
        print(f"Downloaded '{db_file}'.")
    else:
        print(f"'{db_file}' already exists.")

    return

In [20]:
def db_connect():
    # Connect to the Maji Ndogo farm survey database.
    # Make sure the .db file is in the same folder as this notebook.
    connection = sqlite3.connect('Maji_Ndogo_farm_survey_small.db')
    cursor = connection.cursor()

    cursor.execute(
        """SELECT Field_ID, Pollution_level, Plot_size, Annual_yield, Crop_type, Standard_yield
           FROM farm_management_features"""
    )

    # The Crop_type and Annual_yield labels were swapped when the survey was digitised,
    # so we swap them back while building our list of field records.
    field_records = []
    for row in cursor.fetchall():
        field_records.append({
            'Field_ID': row[0],
            'Pollution_level': round(row[1], 4),
            'Plot_size': row[2],
            'Crop_type': row[3],                # stored under 'Annual_yield' in the database
            'Annual_yield': round(row[4], 4),   # stored under 'Crop_type' in the database
            'Standard_yield': round(row[5], 4),
        })

    connection.close()

    print(f"Loaded {len(field_records)} field records.")
    return (field_records,)

In [21]:
db_connect()

Loaded 5654 field records.


([{'Field_ID': 40734,
   'Pollution_level': 0.0853,
   'Plot_size': 1.3,
   'Crop_type': 'cassava',
   'Annual_yield': 0.7514,
   'Standard_yield': 0.578},
  {'Field_ID': 30629,
   'Pollution_level': 0.3997,
   'Plot_size': 2.2,
   'Crop_type': 'cassava',
   'Annual_yield': 1.0699,
   'Standard_yield': 0.4863},
  {'Field_ID': 39924,
   'Pollution_level': 0.358,
   'Plot_size': 3.4,
   'Crop_type': 'tea',
   'Annual_yield': 2.2088,
   'Standard_yield': 0.6496},
  {'Field_ID': 5754,
   'Pollution_level': 0.2867,
   'Plot_size': 2.4,
   'Crop_type': 'cassava',
   'Annual_yield': 1.2776,
   'Standard_yield': 0.5323},
  {'Field_ID': 14146,
   'Pollution_level': 0.0432,
   'Plot_size': 1.5,
   'Crop_type': 'wheat',
   'Annual_yield': 0.8326,
   'Standard_yield': 0.5551},
  {'Field_ID': 5304,
   'Pollution_level': 0.1275,
   'Plot_size': 1.7,
   'Crop_type': 'potato',
   'Annual_yield': 1.1126,
   'Standard_yield': 0.6545},
  {'Field_ID': 429,
   'Pollution_level': 0.0,
   'Plot_size': 2.9,
 

In [9]:
maji_ndogo_sample = [
        {'Field_ID': 40734, 'Pollution_level': 0.0853, 'Plot_size': 1.3, 'Crop_type': 'cassava', 'Annual_yield': 0.7514, 'Standard_yield': 0.578},
        {'Field_ID': 30629, 'Pollution_level': 0.3997, 'Plot_size': 2.2, 'Crop_type': 'cassava', 'Annual_yield': 1.0699, 'Standard_yield': 0.4863},
        {'Field_ID': 39924, 'Pollution_level': 0.358, 'Plot_size': 3.4, 'Crop_type': 'tea', 'Annual_yield': 2.2088, 'Standard_yield': 0.6496},
        {'Field_ID': 5754, 'Pollution_level': 0.2867, 'Plot_size': 2.4, 'Crop_type': 'cassava', 'Annual_yield': 1.2776, 'Standard_yield': 0.5323},
        {'Field_ID': 14146, 'Pollution_level': 0.0432, 'Plot_size': 1.5, 'Crop_type': 'wheat', 'Annual_yield': 0.8326, 'Standard_yield': 0.5551}
    ]

## __Challenge 1: Linear search__

> The survey records are stored in the order the teams submitted them — no sorting, no index. When an inspector calls in a `Field_ID`, we start at record one and check every record until we find a match. This is a **linear search**: `O(n)` — in the worst case, it inspects every record.

### __Task__

Complete `linear_search_field(field_list, target_id)`. It must:
- Iterate through `field_list`.
- Return the `Pollution_level` of the first record whose `Field_ID` matches `target_id`.
- Return `None` if no match is found.

    > ⚠️ Do not change the function name `linear_search_field`.

### __Expected outputs__

- **Input 1:** `linear_search_field(maji_ndogo_sample, 39924)` → `0.358`

- **Input 2:** `print(linear_search_field(maji_ndogo_sample, 99999))` → `None`
    """)

In [3]:
def linear_search_field(field_list, target_id):
    for record in field_list:
        if record.get("Field_ID") == target_id:
            return record.get("Pollution_level")
            
    return None

In [10]:
print(linear_search_field(maji_ndogo_sample, 39924))

0.358


In [11]:
print(linear_search_field(maji_ndogo_sample, 99999))

None


In [21]:
print(linear_search_field(maji_ndogo_sample, 30629))

0.3997


## __Challenge 2: Binary Search__

> If the records are **sorted by `Field_ID`**, we can do much better. Jump to the middle record. Is the target ID higher or lower? Discard half the list. Repeat. This is a **binary search**: `O(log n)` — at one million records, it takes roughly 20 comparisons instead of one million.

The cost: the data must be sorted first.

### __Task__

> Complete `binary_search_yield(sorted_field_list, target_id)`. Assume the input is already sorted by `Field_ID` in ascending order. Return the `Annual_yield` of the matching record, or `None` if not found.

### __Expected outputs__

 > **Input 1:**
    ```python
    sorted_sample = sorted(maji_ndogo_sample, key=lambda f: f['Field_ID'])
    binary_search_yield(sorted_sample, 30629)
    ```
    `1.0699`

> **Input 2:**
    ```python
    sorted_sample = sorted(maji_ndogo_sample, key=lambda f: f['Field_ID'])
    print(binary_search_yield(sorted_sample, 11111))
    ```
    `None`
    """)

In [7]:
def binary_search_yield(sorted_field_list, target_id):
    low = 0
    high = len(sorted_field_list) - 1

    while low <= high:
        mid = (low + high) // 2
        current_id = sorted_field_list[mid]["Field_ID"]

        if current_id == target_id:
            return sorted_field_list[mid].get("Annual_yield")
        elif current_id < target_id:
            low = mid + 1
        else:
            high = mid - 1

    return None

In [8]:
sorted_sample = sorted(maji_ndogo_sample, key=lambda f: f['Field_ID'])
binary_search_yield(sorted_sample, 30629)

1.0699

In [9]:
sorted_sample = sorted(maji_ndogo_sample, key=lambda f: f['Field_ID'])
print(binary_search_yield(sorted_sample, 11111))

None


In [28]:
print(linear_search_field(maji_ndogo_sample, 5754) == binary_search_yield(sorted_sample, 5754))

False


## __Challenge 3: Merge sort__

> Binary search needs sorted data. We need a sorter. Bubble sort is `O(n²)`: double the records, quadruple the work. **Merge sort** is `O(n log n)`: it splits the list in half, sorts each half **recursively**, then merges the two sorted halves.

    Pseudocode:
    ```
    merge_sort(list):
        if len(list) <= 1: return list
        mid   = len(list) // 2
        left  = merge_sort(list[:mid])
        right = merge_sort(list[mid:])
        return merge(left, right)
    ```

### __Task__

> Complete `merge_sort_yields(field_list)` and the helper `merge(left, right)`. Sort records into **descending** order of `Annual_yield` (highest yield first). `merge_sort_yields` must be recursive and must call `merge`.

    > ⚠️ Do not change either function name.

### __Expected outputs__

> **Input 1:** `merge_sort_yields(maji_ndogo_sample)` →
    ```
    [{'Field_ID': 39924, 'Pollution_level': 0.358, 'Plot_size': 3.4, 'Crop_type': 'tea', 'Annual_yield': 2.2088, 'Standard_yield': 0.6496},
     {'Field_ID': 5754, 'Pollution_level': 0.2867, 'Plot_size': 2.4, 'Crop_type': 'cassava', 'Annual_yield': 1.2776, 'Standard_yield': 0.5323},
     {'Field_ID': 30629, 'Pollution_level': 0.3997, 'Plot_size': 2.2, 'Crop_type': 'cassava', 'Annual_yield': 1.0699, 'Standard_yield': 0.4863},
     {'Field_ID': 14146, 'Pollution_level': 0.0432, 'Plot_size': 1.5, 'Crop_type': 'wheat', 'Annual_yield': 0.8326, 'Standard_yield': 0.5551},
     {'Field_ID': 40734, 'Pollution_level': 0.0853, 'Plot_size': 1.3, 'Crop_type': 'cassava', 'Annual_yield': 0.7514, 'Standard_yield': 0.578}]
    ```

> **Input 2:** `[f['Annual_yield'] for f in merge_sort_yields(maji_ndogo_sample)[:3]]` → `[2.2088, 1.2776, 1.0699]`
    """)

In [12]:
def merge(left, right):

    """
    Merges two sorted lists into a single sorted list in descending order of Annual_yield.
    """
    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i]["Annual_yield"] >= right[j]["Annual_yield"]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    # Append any remaining elements
    result.extend(left[i:])
    result.extend(right[j:])

    return result

def merge_sort_yields(field_list):
    """
    Recursively sorts field_list in descending order by Annual_yield using merge sort.
    """
    # Base case: lists of length 0 or 1 are already sorted
    if len(field_list) <= 1:
        return field_list

    # Divide step
    mid = len(field_list) // 2
    left_half = merge_sort_yields(field_list[:mid])
    right_half = merge_sort_yields(field_list[mid:])

    # Conquer/Combine step
    return merge(left_half, right_half)

In [13]:
merge_sort_yields(maji_ndogo_sample)

[{'Field_ID': 39924,
  'Pollution_level': 0.358,
  'Plot_size': 3.4,
  'Crop_type': 'tea',
  'Annual_yield': 2.2088,
  'Standard_yield': 0.6496},
 {'Field_ID': 5754,
  'Pollution_level': 0.2867,
  'Plot_size': 2.4,
  'Crop_type': 'cassava',
  'Annual_yield': 1.2776,
  'Standard_yield': 0.5323},
 {'Field_ID': 30629,
  'Pollution_level': 0.3997,
  'Plot_size': 2.2,
  'Crop_type': 'cassava',
  'Annual_yield': 1.0699,
  'Standard_yield': 0.4863},
 {'Field_ID': 14146,
  'Pollution_level': 0.0432,
  'Plot_size': 1.5,
  'Crop_type': 'wheat',
  'Annual_yield': 0.8326,
  'Standard_yield': 0.5551},
 {'Field_ID': 40734,
  'Pollution_level': 0.0853,
  'Plot_size': 1.3,
  'Crop_type': 'cassava',
  'Annual_yield': 0.7514,
  'Standard_yield': 0.578}]

In [14]:
print([f['Annual_yield'] for f in merge_sort_yields(maji_ndogo_sample)[:3]])

[2.2088, 1.2776, 1.0699]


In [25]:
merge_sort_yields(maji_ndogo_sample)
print(maji_ndogo_sample[0]['Field_ID'])

40734


## __Challenge 4: Lambda and filter — flagging polluted fields__

> Any field above a pollution threshold needs an inspection visit, and the list must update as thresholds change. Python's `filter()` with a `lambda` makes this one readable line.

### __Task__

> Complete `identify_high_pollution(field_list, threshold)`. Use `filter()` and a `lambda` to return only the records whose `Pollution_level` is **strictly greater than** `threshold`. Return the result as a list.

> ⚠️ Do not change the function name `identify_high_pollution`.

### __Expected outputs__

> **Input 1:** `identify_high_pollution(maji_ndogo_sample, 0.3)` →
    ```python
    [{'Field_ID': 30629, 'Pollution_level': 0.3997, 'Plot_size': 2.2, 'Crop_type': 'cassava', 'Annual_yield': 1.0699, 'Standard_yield': 0.4863},
     {'Field_ID': 39924, 'Pollution_level': 0.358, 'Plot_size': 3.4, 'Crop_type': 'tea', 'Annual_yield': 2.2088, 'Standard_yield': 0.6496}]
    ```

> **Input 2:** `identify_high_pollution(maji_ndogo_sample, 0.5)` → `[]`
    """)

In [7]:
def identify_high_pollution(field_list, threshold):
    """
    Filters field_list for records where Pollution_level is strictly greater than threshold.
    """
    return list(filter(lambda record: record.get("Pollution_level", 0) > threshold, field_list))

In [10]:
identify_high_pollution(maji_ndogo_sample, 0.3)

[{'Field_ID': 30629,
  'Pollution_level': 0.3997,
  'Plot_size': 2.2,
  'Crop_type': 'cassava',
  'Annual_yield': 1.0699,
  'Standard_yield': 0.4863},
 {'Field_ID': 39924,
  'Pollution_level': 0.358,
  'Plot_size': 3.4,
  'Crop_type': 'tea',
  'Annual_yield': 2.2088,
  'Standard_yield': 0.6496}]

In [11]:
identify_high_pollution(maji_ndogo_sample, 0.5)

[]

In [12]:
identify_high_pollution(maji_ndogo_sample, 0.3997)

[]

## __Challenge 5: Lambda and map — projecting harvests__

> If `filter()` decides which items to *keep*, `map()` transforms *every* item. The Ministry of Agriculture wants a quick harvest projection for each field: `Plot_size` × `Standard_yield`, rounded to 2 decimal places.

### __Task__

> Complete `project_harvests(field_list)`. Use `map()` and a `lambda` to compute each field's projected yield (`Plot_size * Standard_yield`, rounded to 2 decimals). Return the result as a list.

> ⚠️ Do not change the function name `project_harvests`.

### __Expected outputs__

> **Input 1:** `project_harvests(maji_ndogo_sample)` → `[0.75, 1.07, 2.21, 1.28, 0.83]`

> **Input 2:** `project_harvests(maji_ndogo_sample[2:4])` → `[2.21, 1.28]`
    """)

In [14]:
def project_harvests(field_list):
    """
    Computes projected yield (Plot_size * Standard_yield) rounded to 2 decimal places
    for each record in field_list using map() and a lambda function.
    """
    return list(
            map(lambda record: round(record.get("Plot_size", 0) * record.get("Standard_yield", 0), 2),
                field_list
            )
    )

In [15]:
project_harvests(maji_ndogo_sample)

[0.75, 1.07, 2.21, 1.28, 0.83]

In [17]:
project_harvests(maji_ndogo_sample[2:4])

[2.21, 1.28]

In [18]:
project_harvests(maji_ndogo_sample[1:3])

[1.07, 2.21]